# Aula 20: Codificação de Huffman — Compressão de Telemetria do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, exploraremos a aplicação do **Algoritmo de Huffman** (uma árvore binária de busca baseada em frequência) para realizar a compressão sem perdas (*lossless*) dos pacotes de telemetria transmitidos entre o AGV e o sistema supervisor SCADA.

### Contexto no Projeto Visão-AGV

Durante a operação autônoma, o AGV envia continuamente pacotes de dados contendo estados operacionais como:
- `EM_TRANSITO`, `AGUARDANDO_CRUZAMENTO`, `CARREGANDO_BATERIA`, `BATERIA_BAIXA`, `DESVIO_OBSTACULO`, `PARADA_EMERGENCIA`

Em redes industriais sem fio sujeitas a ruídos e largura de banda limitada, utilizar codificação de tamanho fixo (ex: ASCII de 8 bits por caractere) desperdiça recursos. Como certos estados (como `EM_TRANSITO`) ocorrem com frequência muito maior que outros (como `PARADA_EMERGENCIA`), a **Codificação de Huffman** atribui códigos de comprimento variável: símbolos mais frequentes recebem sequências de bits mais curtas, reduzindo o tamanho total transmitido e a latência da rede.

--- 

## 2. Estrutura do Notebook

1. **Geração da Telemetria Sintética**: Simulação de um fluxo de registros de estado do AGV.
2. **Construção da Árvore de Huffman**: Implementação da estrutura de nó de árvore e uso de Min-Heap (`heapq`).
3. **Tabela de Prefixos**: Geração dos códigos binários livres de prefixo.
4. **Codificação e Decodificação**: Codificação do fluxo de telemetria em binário e decodificação de volta ao texto original.
5. **Análise de Eficiência**: Comparação do tamanho em bits original vs. comprimido e taxa de compressão.


In [1]:
# Setup Inicial e Simulação de Log de Telemetria do AGV
from collections import Counter
import heapq

# Simulação de um fluxo real de telemetria emitido pelo AGV durante um turno de trabalho
# Note a alta frequência de 'EM_TRANSITO' e a baixa frequência de eventos raros como 'PARADA_EMERGENCIA'
log_telemetria = (
    "EM_TRANSITO;EM_TRANSITO;EM_TRANSITO;EM_TRANSITO;"
    "AGUARDANDO_CRUZAMENTO;EM_TRANSITO;EM_TRANSITO;"
    "CARREGANDO_BATERIA;EM_TRANSITO;EM_TRANSITO;"
    "DESVIO_OBSTACULO;EM_TRANSITO;EM_TRANSITO;"
    "BATERIA_BAIXA;CARREGANDO_BATERIA;EM_TRANSITO;"
    "PARADA_EMERGENCIA;EM_TRANSITO;EM_TRANSITO"
)

print(f"Tamanho do texto original: {len(log_telemetria)} caracteres")
print(f"Tamanho estimado em ASCII (8 bits/char): {len(log_telemetria) * 8} bits")

# Contagem de frequências dos caracteres
frequencias = Counter(log_telemetria)
print("\nFrequência dos caracteres no log de telemetria:")
for char, freq in sorted(frequencias.items(), key=lambda x: x[1], reverse=True):
    char_repr = repr(char)
    print(f"  Caractere {char_repr:10s} -> Frequência: {freq}")


## 3. Construção da Árvore de Huffman

A Árvore de Huffman é construída de baixo para cima (bottom-up):
1. Cada caractere inicia como um nó folha contendo sua frequência.
2. Os dois nós de menor frequência são combinados em um nó pai cuja frequência é a soma de ambas.
3. O processo se repete com o Min-Heap até restar apenas um nó (a raiz da árvore).


In [2]:
# Estrutura do Nó da Árvore de Huffman
class NoHuffman:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.esquerda = None
        self.direita = None

    # Permite comparação no Min-Heap com base na frequência
    def __lt__(self, outro):
        return self.freq < outro.freq

def construir_arvore_huffman(frequencias):
    """
    Constrói a Árvore de Huffman a partir do dicionário de frequências dos símbolos.
    """
    heap = [NoHuffman(char, freq) for char, freq in frequencias.items()]
    heapq.heapify(heap)

    # Combina iterativamente os dois nós com as menores frequências
    while len(heap) > 1:
        no1 = heapq.heappop(heap)
        no2 = heapq.heappop(heap)

        pai = NoHuffman(None, no1.freq + no2.freq)
        pai.esquerda = no1
        pai.direita = no2

        heapq.heappush(heap, pai)

    return heap[0] if heap else None

# Construção da árvore
raiz_huffman = construir_arvore_huffman(frequencias)
print("Árvore de Huffman construída com sucesso!")


## 4. Geração dos Códigos de Huffman (Tabela de Prefixos)

Ao percorrer a árvore da raiz até cada folha:
- Adiciona-se `'0'` ao ir para a esquerda.
- Adiciona-se `'1'` ao ir para a direita.

Como os símbolos ficam exclusivamente nas folhas, nenhum código é prefixo de outro, garantindo unicos decodificáveis (*prefix-free code*).


In [3]:
def gerar_tabela_codigos(raiz, codigo_atual="", tabela=None):
    """
    Percorre a Árvore de Huffman recursivamente para gerar a tabela de códigos binários.
    """
    if tabela is None:
        tabela = {}

    if raiz is not None:
        # Se for um nó folha, registra o código correspondente
        if raiz.char is not None:
            tabela[raiz.char] = codigo_atual
        gerar_tabela_codigos(raiz.esquerda, codigo_atual + "0", tabela)
        gerar_tabela_codigos(raiz.direita, codigo_atual + "1", tabela)

    return tabela

tabela_huffman = gerar_tabela_codigos(raiz_huffman)

print("--- TABELA DE CÓDIGOS DE HUFFMAN ---")
for char, codigo in sorted(tabela_huffman.items(), key=lambda item: len(item[1])):
    char_repr = repr(char)
    freq = frequencias[char]
    print(f"Símbolo {char_repr:10s} | Freq: {freq:3d} | Código Huffman: {codigo}")


## 5. Codificação e Decodificação do Log de Telemetria

Agora testamos a compressão substituindo cada símbolo pelo seu código de Huffman e, em seguida, realizamos a reconstrução exata do log utilizando a árvore de busca.

In [4]:
def codificar_mensagem(mensagem, tabela):
    """Converte o texto original em uma string binária comprimida."""
    return "".join(tabela[char] for char in mensagem)

def decodificar_mensagem(bits_comprimidos, raiz):
    """Reconstrói o texto original percorrendo a árvore nó a nó."""
    mensagem_decodificada = []
    atual = raiz
    
    for bit in bits_comprimidos:
        if bit == '0':
            atual = atual.esquerda
        else:
            atual = atual.direita
            
        # Quando atinge um nó folha, extrai o caractere e retorna à raiz
        if atual.char is not None:
            mensagem_decodificada.append(atual.char)
            atual = raiz
            
    return "".join(mensagem_decodificada)

# Teste de Codificação
telemetria_comprimida = codificar_mensagem(log_telemetria, tabela_huffman)
telemetria_restaurada = decodificar_mensagem(telemetria_comprimida, raiz_huffman)

print("Compressão realizada com sucesso!")
print(f"Amostra binária comprimida (primeiros 60 bits): {telemetria_comprimida[:60]}...")
print(f"Validação de integridade (Decodificação == Original): {telemetria_restaurada == log_telemetria}")


## 6. Análise de Eficiência e Impacto na Banda RF do AGV

Avaliamos o ganho percentual de taxa de compressão e a redução de payload na comunicação sem fio.

In [5]:
tamanho_original_bits = len(log_telemetria) * 8
tamanho_comprimido_bits = len(telemetria_comprimida)
taxa_compressao = (1 - (tamanho_comprimido_bits / tamanho_original_bits)) * 100
bits_por_simbolo_medio = tamanho_comprimido_bits / len(log_telemetria)

print("=== ANÁLISE DE DESEMPENHO DA COMPRESSÃO DE HUFFMAN ===")
print(f"Tamanho Original (ASCII 8-bit) : {tamanho_original_bits} bits ({len(log_telemetria)} bytes)")
print(f"Tamanho Comprimido (Huffman)   : {tamanho_comprimido_bits} bits ({tamanho_comprimido_bits / 8:.2f} bytes)")
print(f"Média de bits por caractere    : {bits_por_simbolo_medio:.2f} bits/char (Economia em relação aos 8 bits padrão)")
print(f"Redução total de banda (RF)   : {taxa_compressao:.2f}%")

print("\n--- CONCLUSÃO PARA O PROJETO VISÃO-AGV ---")
print("1. O algoritmo de Huffman reduziu o volume de dados transmitidos pelo módulo sem fio do AGV em mais de 40%.")
print("2. Caracteres recorrentes como 'E', 'M', '_', 'T' receberam códigos muito curtos, enquanto eventos raros como 'P' e 'C' não oneraram o tráfego regular.")
print("3. Isso garante maior estabilidade na telemetria SCADA em ambientes de fábrica com alta interferência eletromagnética.")
